# 02 — Generative AI Concepts, No LangChain
Calling the corrected `ask()` helper directly so you see exactly what each prompting technique does to the request, with nothing hidden by a framework.

# Setup
Run this first in every notebook. It assumes this notebook lives in a folder
that can reach `inhouse_wrappers.py` (the CORRECTED version, from `wrapper_fix/`),
`rag_pure_python.py`, and `inhouse_llm.py`. Adjust the `sys.path.append(...)`
lines below if your folder layout differs.

**Corrected in this version:** uses `ask()`/`ask_vision()` (built on the fixed
`get_chat_model()`) instead of calling `multimodal_chat()` directly — the
original always hit the Qwen3-14B endpoint regardless of which `model=` you
asked for. Embeddings go through `embedder.embed_query()`/`.embed_documents()`
instead of `get_embedding(text, model=MODEL_JINA)`, which doesn't match the
real function signature in your `inhouse_llm.py` (no `model=` kwarg there).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../wrapper_fix"))           # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("."))                          # folder containing inhouse_llm.py / rag_pure_python.py
# sys.path.append("/path/to/inhouse_rag_capstone")              # uncomment & adjust if needed

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B, MODEL_JINA
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. Zero-shot vs Few-shot
**Why:** few-shot examples steer output *format* and *style* far more reliably than instructions alone, especially for classification/extraction tasks. **When:** use few-shot whenever you need consistent structured output and zero-shot isn't reliable enough; skip it for simple open-ended generation where it just wastes tokens.

(`ask()` is already defined in the setup cell above, correctly routed per model.)

In [ ]:
zero_shot = ask(
    "Classify sentiment as positive, negative, or neutral. Reply with one word.",
    "The model latency improved a lot after the upgrade."
)

few_shot = ask(
    "Classify sentiment as positive, negative, or neutral. Reply with one word.\n\n"
    "Text: The service was terrible.\nLabel: negative\n\n"
    "Text: It was an average experience.\nLabel: neutral\n\n"
    "Text: Absolutely loved it!\nLabel: positive\n",
    "Text: The model latency improved a lot after the upgrade.\nLabel:"
)

print("Zero-shot:", zero_shot)
print("Few-shot :", few_shot)

## 2. Chain-of-thought (CoT) prompting
**Why:** asking the model to reason step-by-step before answering improves accuracy on multi-step problems (math, logic, multi-hop questions). **When:** use it for anything requiring intermediate reasoning; skip it for simple lookups — it just adds latency/tokens with no benefit.

In [ ]:
direct = ask("Answer with only the final number.",
             "A retriever returns 5 chunks. 2 are irrelevant. Of the relevant ones, "
             "half mention MCP. How many chunks mention MCP?")

cot = ask("Think step by step, then give the final answer on the last line as 'Answer: <n>'.",
          "A retriever returns 5 chunks. 2 are irrelevant. Of the relevant ones, "
          "half mention MCP. How many chunks mention MCP?")

print("Direct:\n", direct)
print("\nCoT:\n", cot)

## 3. Structured output (JSON) without a parsing library
**Why:** downstream code needs reliable structure, not prose. **When:** any time an LLM output feeds into another system (a UI, a database row, another API call) — this is the bridge between 'generative AI' and 'software'.

In [ ]:
import json

raw = ask(
    "Respond with ONLY valid JSON, no markdown fences, no extra text. "
    "Schema: {\"model\": str, \"use_case\": str, \"confidence\": float}",
    "Recommend which in-house model fits 'summarizing a 50-page PDF'."
)
print("Raw:", raw)

try:
    parsed = json.loads(raw)
    print("Parsed OK:", parsed)
except json.JSONDecodeError:
    print("Model didn't return clean JSON — this is exactly why output parsers "
          "(see notebook 03) exist as a safety net.")

## 4. Summarization, extraction, rewriting — the generic-task triad
These three task shapes cover most production GenAI use cases. Notice the prompt pattern is always: role/constraint in system, content + instruction in user.

In [ ]:
text = (docs := """
Model Context Protocol (MCP) standardizes how LLMs call external tools and data
sources through a client-server architecture. It separates the model from the
tool implementation, so any compliant client can use any compliant server.
""")

summary = ask("Summarize in one sentence.", text)
extraction = ask("Extract the key entities mentioned, as a comma-separated list.", text)
rewrite = ask("Rewrite this for a 10-year-old.", text)

print("Summary:", summary)
print("Entities:", extraction)
print("Simplified:", rewrite)

## 5. Model comparison — same prompt, different in-house models
**Why practice this:** picking a model isn't theoretical — run the same task across your available models and *look at* the differences before committing.

**Important:** under the OLD `multimodal_chat()`-based code, this may have silently queried Qwen3-14B for every row regardless of `model=`. With `ask()`, each row genuinely hits its own model's endpoint.

In [ ]:
prompt = "Explain the difference between RAG and fine-tuning in 2 sentences."
for name, model in [("Qwen3-14B", MODEL_QWEN3_14B),
                     ("Qwen3-30B", MODEL_QWEN3_30B),
                     ("Mistral",   MODEL_MISTRAL)]:
    print(f"--- {name} ---")
    print(ask("Be concise.", prompt, model=model, max_tokens=150))
    print()